# The Allegheny Family Screening Tool: When Being Poor Looks Like Being Risky

In this notebook I look at the Allegheny Family Screening Tool, a child welfare risk-prediction system, as a twelfth mechanism behind an AI ethics failure: a model that can only see a family's involvement with public systems, deployed on a population where being poor is exactly what determines whether that involvement is public at all.

In this notebook, I will:

- Summarize what the tool does and the core criticism child welfare researchers and journalists have raised about it
- Simulate two income groups with an identical real underlying rate of a risk-relevant issue, differing only in which systems, public or private, they resolve it through
- Measure how that one difference in visibility produces a huge gap in both false alarms and missed real risk
- Try the fix critics specifically called for, removing the clearest poverty-only variable, and see that it closes one gap while leaving the other almost untouched
- Sit with why this particular mechanism doesn't have a clean technical fix, only mitigations

## 1. Background: What Was Reported

Starting in August 2016, Allegheny County, Pennsylvania, began using a predictive risk model, the Allegheny Family Screening Tool, to help call screeners at its child welfare hotline decide whether a report of suspected child maltreatment should be investigated in person. The tool generates a numeric risk score for each report, built from the county's integrated data warehouse, a system that links records from child welfare history, public behavioral health and substance use treatment, county jail bookings, and public benefits programs like food and cash assistance.

A central criticism, raised by researcher Virginia Eubanks in her book Automating Inequality and later echoed by journalists and academic reviewers who examined the tool, is about what that data warehouse can and cannot see. It only contains records generated by contact with public systems. A family that gets mental health care, substance use treatment, or other support through private insurance and private providers leaves no trace in it at all, regardless of what's actually happening in that family. A family that relies on public services for the same underlying needs, often because they have no other option, shows up in exactly the records the model is built from. The county has defended the tool, pointed out that it does not use race as an input, and revised it across versions, removing some variables and keeping a human screener in the loop with the authority to override the score. Independent evaluators and journalists examining the tool, including later investigations by the Associated Press and other newsrooms, have continued to find that families from lower-income and predominantly Black neighborhoods are flagged for investigation at disproportionately high rates relative to their actual outcomes.

## 2. Why This Is a Different Mechanism

This isn't a proxy feature quietly standing in for a protected attribute inside an otherwise representative dataset, and it isn't an explicit attribute typed directly into a score. It's a gap in what the data can see at all, one that lines up with income for a structural reason rather than a modeling choice: a family's economic situation is exactly what decides whether getting help for a given problem happens through a channel the county's data warehouse records, or through one it has no way to observe.

The same underlying issue, say a parent's substance use or a mental health crisis, can occur at the same rate in a wealthy household and a poor one. If the wealthy household resolves it privately, nothing about it ever reaches this system. If the poor household resolves it publicly, because that's the only option available to them, it becomes part of a risk score. The tool doesn't need to know anyone's income to produce this gap. Income decides which channel a family uses, and the model only has eyes on one of those channels.

## 3. A Note on the Simulation Below

This notebook does not use Allegheny County's real model, real variables, or real score thresholds, none of which are something I should try to reproduce. I use a small set of simplified attributes, whether a family has an underlying issue, whether it gets resolved publicly or privately, and whether the family separately receives a general public benefit unrelated to that issue, to make one mechanism concrete: how a model built only from public-system records can end up scoring poverty itself as risk, while remaining nearly blind to the same real issue in a wealthier household.

## 4. Simulating Two Income Groups With the Same True Rate of a Real Issue

I give both groups the exact same underlying rate of a risk-relevant issue, since conditions like substance use and mental health struggles occur across income levels at broadly similar rates in practice. The groups differ only in two ways that reflect how income actually shapes which systems a family relies on: how often a real issue gets resolved through a publicly recorded channel rather than a private one, and how often the family separately receives a general public benefit that has nothing to do with the issue at all.

In [ ]:
import random

random.seed(17)

N_PER_GROUP = 5000
TRUE_ISSUE_RATE = 0.15

GROUPS = [
    {"name": "Lower-income families", "resolution_public_rate": 0.90, "benefit_baseline_rate": 0.70},
    {"name": "Higher-income families", "resolution_public_rate": 0.05, "benefit_baseline_rate": 0.01},
]


def make_families(n, true_issue_rate, resolution_public_rate, benefit_baseline_rate):
    """Returns a list of family dicts with a true issue flag and two separate public-record flags."""
    families = []
    for _ in range(n):
        has_issue = random.random() < true_issue_rate
        resolved_publicly = has_issue and random.random() < resolution_public_rate
        receives_public_benefit = random.random() < benefit_baseline_rate
        families.append({
            "has_issue": has_issue,
            "resolved_publicly": resolved_publicly,
            "receives_public_benefit": receives_public_benefit,
        })
    return families

## 5. Building a Risk Score From Public Records Alone

The score below counts only what shows up in a public record: whether the family's issue, if they have one, was resolved through a publicly visible channel, and whether the family separately receives a general public benefit. Nothing here ever looks at income directly, and nothing here can see a privately resolved issue at all.

In [ ]:
def compute_risk_score(family):
    """Returns a simple risk score built only from a family's public-record touchpoints."""
    return int(family["resolved_publicly"]) + int(family["receives_public_benefit"])


for group in GROUPS:
    group["families"] = make_families(
        N_PER_GROUP, TRUE_ISSUE_RATE, group["resolution_public_rate"], group["benefit_baseline_rate"]
    )
    for family in group["families"]:
        family["risk_score"] = compute_risk_score(family)

    avg_score = sum(f["risk_score"] for f in group["families"]) / len(group["families"])
    print(group["name"], "- average risk score:", round(avg_score, 3))

## 6. Measuring the Detection Gap

I check two things for each group, using a report as "flagged" once it has any public-record touchpoint at all: the recall, how often a family with a genuine real issue actually gets flagged, and the false positive rate, how often a family with no real issue at all still gets flagged, purely from the public benefit touchpoint.

In [ ]:
def recall_and_false_positive_rate(families, flag_fn):
    """Returns the recall among families with a real issue and the false positive rate among those without one."""
    with_issue = [f for f in families if f["has_issue"]]
    without_issue = [f for f in families if not f["has_issue"]]
    recall = len([f for f in with_issue if flag_fn(f)]) / len(with_issue)
    false_positive_rate = len([f for f in without_issue if flag_fn(f)]) / len(without_issue)
    return recall, false_positive_rate


flag_on_full_score = lambda f: f["risk_score"] >= 1

for group in GROUPS:
    recall, fpr = recall_and_false_positive_rate(group["families"], flag_on_full_score)
    print(group["name"])
    print("  recall (flagged among families with a real issue):", round(recall, 3))
    print("  false positive rate (flagged among families with no issue):", round(fpr, 3))

## 7. Trying the Fix Critics Specifically Called For

Critics of the tool have specifically pointed to variables tied to general public benefit receipt as the clearest case of poverty itself being scored as risk. I try removing that one variable entirely, flagging a family only based on whether their issue, if they have one, was resolved publicly, and recheck both numbers.

In [ ]:
flag_on_resolution_only = lambda f: f["resolved_publicly"]

for group in GROUPS:
    recall, fpr = recall_and_false_positive_rate(group["families"], flag_on_resolution_only)
    print(group["name"])
    print("  recall after removing the benefit variable:", round(recall, 3))
    print("  false positive rate after removing the benefit variable:", round(fpr, 3))

## 8. What the Fix Closes, and What It Leaves Standing

Removing the benefit variable eliminates the false positive rate entirely for both groups, a family with no real issue and no public benefit record now has nothing in the score to flag at all, a real and meaningful improvement. But the recall gap between the two groups barely moves. Higher-income families with a genuine real issue are still missed at nearly the same rate as before, because the actual mechanism driving that gap was never the benefit variable, it was the fact that their issue almost never shows up in a publicly recorded channel in the first place. Removing one variable fixes the part of the problem that variable was causing, and this gap was never that part.

## 9. Why This Mechanism Doesn't Have a Clean Technical Fix

Every earlier notebook in this series eventually found some change, a dropped feature, a cross-checked sensor, a different policy, that closed the gap it measured. This one is different: closing the recall gap directly would mean giving the model visibility into private healthcare and private treatment records it currently can't see at all, which trades one serious harm, a biased score, for another, dramatically expanding surveillance into parts of people's lives specifically because they can afford to keep them private. There may not be a version of this tool that is both accurate across income levels and not that invasive. What Allegheny County has actually done is narrower: remove the clearest poverty-only variables across tool versions, keep a trained human screener able to override the score rather than let it decide automatically, and commission independent evaluations, mitigations aimed at limiting the damage of a gap that the tool's basic design can't fully close on its own.

## 10. What This Costs in Real Terms

Every number in this notebook so far has been a rate out of five thousand simulated families. In the real system, a flagged report can lead to an in-person investigation of a family, with everything that entails for a parent and child, triggered in part by nothing more than that family's economic reliance on public assistance. Researchers studying the tool have also raised a quieter cost: if poor families come to believe that using public mental health or substance use services raises their risk of a child welfare investigation, some may avoid seeking help they and their children actually need, purely to stay off a system's radar, which would make the underlying problem the tool is trying to catch worse, not better.

## 11. Lessons for Building AI Systems

What I take from working through this:

- A model is only as blind or as sighted as the data sources feeding it, and if those sources are tied to one specific channel, public systems, private systems, a particular platform, the model will systematically miss or overcount whatever determines which channel someone happens to use.
- When that channel choice itself correlates with income, or any other circumstance someone doesn't choose freely, the resulting score isn't measuring the thing it claims to measure, it's measuring exposure to one kind of institution.
- Not every mechanism in this series has had a clean fix. Some only have mitigations, and it's worth being honest in the model's own documentation about which kind of problem a given gap actually is, rather than implying a future software update will resolve it.
- Removing one variable can close one specific leak without touching the structural reason the leak existed, and declaring success after the easy fix is itself a way gaps like this survive multiple rounds of well-intentioned revision.

## 12. Conclusion and Next Steps

In this notebook I used a synthetic simulation to work through how a risk model built entirely from public-system records can end up scoring poverty itself as risk, flagging families with no real issue at a high rate purely for needing public assistance, while missing the same real issue in wealthier families almost entirely, because it never occurs anywhere the model can see. Removing the clearest poverty-only variable fixed the false alarms and left the deeper visibility gap almost exactly where it started.

Across the twelve notebooks in this series now, most mechanisms eventually met some fix that closed the gap they measured. This one is a reminder that isn't always true: some gaps are downstream of what a system is structurally allowed to see, and the honest next step isn't always a better model, sometimes it's naming the limits of what the available data could ever support, and building the human and policy safeguards around that limit instead of past it.

I want to leave Virginia Eubanks's Automating Inequality and the independent evaluations and journalism that have followed the Allegheny Family Screening Tool across its several versions as further reading rather than trying to simplify them any further here, this notebook was only ever meant to make one mechanism concrete, not to stand in for either.